<a href="https://colab.research.google.com/github/Malik-Zain-555/resume-category-classifier/blob/main/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
path = "/content/drive/MyDrive/Datasets - ML/Resume.csv"

df = pd.read_csv(path)

df.head(2)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR


In [2]:
import re
def resume_cleaner(text):

  text = text.lower()

  text = re.sub(r'http\S+|www\.\S+', " ", text)

  text = re.sub(r'\S+@\S+', " ", text)

  text = re.sub(r'[^a-zA-Z\s]', " ", text)

  text = re.sub(r'\s+'," ", text).strip()

  return text

In [3]:
df['Resume_str'] = df['Resume_str'].apply(resume_cleaner)

df.head(1)

,ID,Resume_str,Resume_html,Category
0,16852973,hr administrator marketing associate hr admini...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
x = df['Resume_str']
y = df['Category']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print(x_train.shape)
print(x_test.shape)

(1987,)
(497,)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score


pl = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1,2))),
    ('logReg', LogisticRegression(max_iter=1000, random_state=42, C=1000))
])

score = cross_val_score(pl, x_train, y_train ,cv=5, n_jobs=1)

print(score.mean(), score.std())

0.6899700011391972 0.03279618570242763


# **Embedding**

In [6]:
!pip install sentence-transformers          # library install karo
from sentence_transformers import SentenceTransformer, util   # use import karo
model = SentenceTransformer('all-MiniLM-L6-v2')   # pehle se trained model download karo (ek baar)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
sents = ["frontend developer", "web developer", "chef"]   # 3 jumle
emb = model.encode(sents)                   # jumle numbers mein badlo

print(util.cos_sim(emb[0], emb[1]))         # frontend developer vs web developer
print(util.cos_sim(emb[0], emb[2]))         # frontend developer vs chef

tensor([[0.6632]])
tensor([[0.1076]])


In [8]:
sents.append("teacher")
emb = model.encode(sents)

print(util.cos_sim(emb[0], emb[1]))
print(util.cos_sim(emb[0], emb[2]))
print(util.cos_sim(emb[0], emb[3]))
print(util.cos_sim(emb[1], emb[3]))

tensor([[0.6632]])
tensor([[0.1076]])
tensor([[0.1816]])
tensor([[0.3618]])


In [9]:
print(emb.shape)
print(emb[0][:5])

(4, 384)
[-0.02335857 -0.05619968 -0.03134041 -0.00634115  0.05395032]


In [10]:
small = x_train.tolist()[:5]
small_emb = model.encode(small)
print(small_emb.shape)

(5, 384)


In [11]:
x_train_emb = model.encode(x_train.tolist(), show_progress_bar=True)
x_test_emb = model.encode(x_test.tolist(), show_progress_bar=True)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [12]:
print(x_train_emb.shape)
print(x_test_emb.shape)

(1987, 384)
(497, 384)


In [13]:
clf = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_val_score(clf, x_train_emb, y_train, cv=5, n_jobs=1)
print(scores.mean(), scores.std())

0.680426059769882 0.03173971978954608


In [14]:
for C in [1,10, 100, 1000]:
  clf = LogisticRegression(C=C, random_state=42, max_iter=1000)
  scores = cross_val_score(clf, x_train_emb, y_train, cv=5, n_jobs=1)
  print(C, scores.mean().round(4), scores.std().round(4))

1 0.6804 0.0317
10 0.7061 0.0352
100 0.6819 0.0216
1000 0.6759 0.0203


In [15]:
clf = LogisticRegression(C=10, max_iter=1000, random_state=42)
clf.fit(x_train_emb, y_train)

resume = "I'm Zain. I have 3+ year experience of web dev as a Frontend and I did backend and database. Used Mongo DB and mySQL and other stuff."
emb = model.encode([resume_cleaner(resume)])

print(clf.predict(emb))
print(clf.predict_proba(emb).max())

['INFORMATION-TECHNOLOGY']
0.5317418214399344


In [16]:
dental = "Hi, I'm ZIKAKITO. I have 13+ experience in dental field. I am best doctor in pakistan."
emb = model.encode([resume_cleaner(dental)])

print(clf.predict(emb))
print(clf.predict_proba(emb).max())

['HEALTHCARE']
0.21962495125521123


In [17]:
chef = "**Professional Summary – Chef** Dedicated and skilled Chef with experience in kitchen operations, food preparation, and menu planning.  Proficient in culinary techniques, food presentation, and maintaining high food quality standards.  Experienced in managing kitchen staff and ensuring smooth restaurant operations.  Knowledgeable in food safety, hygiene, and inventory management.  Passionate about creating delicious dishes and delivering excellent customer satisfaction.  Committed to maintaining cleanliness, teamwork, and efficiency in a professional kitchen."
hr = "**Professional Summary – Human Resources (HR)** Dedicated HR professional with knowledge of recruitment, employee relations, and human resource management.  Experienced in payroll processing, attendance management, and maintaining employee records.  Skilled in interviewing, hiring, onboarding, and staff coordination.  Proficient in handling employee concerns and supporting workplace policies.  Knowledgeable in HR documentation, performance management, and labor regulations.  Committed to improving employee satisfaction and supporting organizational growth."

emb = model.encode([resume_cleaner(chef)])

print(clf.predict(emb))
print(clf.predict_proba(emb).max())

['CHEF']
0.9868178055420738


In [18]:
emb = model.encode([resume_cleaner(hr)])

print(clf.predict(emb))
print(clf.predict_proba(emb).max())

['HR']
0.9880798268084768


In [19]:
print(clf.score(x_test_emb, y_test))

0.744466800804829


In [20]:
clf = LogisticRegression(C=10, max_iter=1000, random_state=42)
clf.fit(x_train_emb, y_train)

print("train:", clf.score(x_train_emb, y_train))
print("test:", clf.score(x_test_emb, y_test))
print(x_test_emb.shape, len(y_test))

train: 0.9018621036738802
test: 0.744466800804829
(497, 384) 497
